# 03 - YOLO inference wiring (T5, issue #6)

Live bridge between camera intake (T1) and pure scoring (T2): load
`yolo26n-pose.pt` once, run per-frame inference, hand the scorer numpy
`(17, 2)` keypoints plus `(17,)` visibility with box areas. Torch lives only
at this edge (`np.asarray` on call); scoring never sees tensors. Empty
frames yield `[]`; failures raise `InferenceError` carrying a machine code
the T3 loop emits as the frozen `error` event. Person selection stays T2's
`select_largest_person` (no duplication). Weights stay out of the repo —
default is the official name so any machine auto-downloads; fallback list
per ADR-0003. Device resolves cuda -> mps -> cpu, bound once at load
(explicit device wins); the setup report feeds the version record.
Eval/no-grad/letterboxing/NMS live inside Ultralytics predict — the
abstraction boundary, correctly untested here; never re-implement by hand.
No per-frame identity tracking (single-desk-worker scope): revisit via
Ultralytics model.track only if largest-Person flapping is observed live.
Deliberate non-goals: annotated-media export (forbidden ADR-0002), file
input (out ADR-0001), small-object slicing and line counting (N/A).
No persistence (ADR-0002).

In [ ]:
import numpy as np

In [ ]:
MODEL_NAME = "yolo26n-pose.pt"
FALLBACK_MODELS = ("yolo11n-pose.pt",)


def resolve_device():
    """cuda -> mps -> cpu, mirroring the reference example's chain."""
    import torch

    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


class InferenceError(Exception):
    """Inference failure carrying a machine-readable code."""

    def __init__(self, code, message):
        super().__init__(message)
        self.code = code
        self.message = message


def load_pose_model(path=MODEL_NAME, fallbacks=FALLBACK_MODELS, loader=None, device=None):
    """Load weights once, bound to device; fall back per ADR-0003.

    Returns (model, report); report carries model name, task, device for
    the version record. Ultralytics predict runs no-grad with internal
    letterboxing/NMS, so no manual eval/preprocess step is needed.
    """
    if loader is None:
        from ultralytics import YOLO

        loader = YOLO
    resolved = device or resolve_device()
    tried = [path, *fallbacks]
    last_error = None
    for name in tried:
        try:
            model = loader(name, task="pose", verbose=False)
            model.to(resolved)
            return model, {"model": name, "task": "pose", "device": resolved}
        except Exception as exc:
            last_error = exc
    raise InferenceError(
        "WEIGHTS_UNAVAILABLE", f"none of {tried} could be loaded: {last_error}"
    )

In [ ]:
def extract_persons(result):
    """One YOLO result -> [{area, xyn (17,2), visibility (17,)}] as numpy."""
    keypoints = getattr(result, "keypoints", None)
    if keypoints is None:
        return []
    xyn = np.asarray(keypoints.xyn, dtype=float)
    data = np.asarray(keypoints.data, dtype=float)
    if xyn.shape[0] == 0:
        return []
    visibility = data[:, :, 2]
    boxes = getattr(result, "boxes", None)
    box_coords = getattr(boxes, "xyxy", None) if boxes is not None else None
    xyxy = np.asarray(box_coords, dtype=float) if box_coords is not None else None
    persons = []
    for index in range(xyn.shape[0]):
        if xyxy is not None and index < xyxy.shape[0]:
            x1, y1, x2, y2 = xyxy[index]
            area = float(max(0.0, x2 - x1) * max(0.0, y2 - y1))
        else:
            spread_x = xyn[index, :, 0]
            spread_y = xyn[index, :, 1]
            area = float(
                max(0.0, spread_x.max() - spread_x.min())
                * max(0.0, spread_y.max() - spread_y.min())
            )
        persons.append(
            {"area": area, "xyn": xyn[index], "visibility": visibility[index]}
        )
    return persons


def infer_frame(model, frame):
    """Run one frame -> persons list; failures become InferenceError."""
    try:
        results = model(frame, verbose=False)
    except InferenceError:
        raise
    except Exception as exc:
        raise InferenceError(
            "INFERENCE_FAILED", str(exc) or type(exc).__name__
        )
    if not results:
        return []
    return extract_persons(results[0])

In [ ]:
# LIVE: load weights and run one real frame. Manual run with BALDUR_LIVE=1;
# skipped by the test loader and by headless verify (variable unset).
import os

if os.getenv("BALDUR_LIVE") == "1":
    import cv2

    _capture = cv2.VideoCapture(0)
    if _capture.isOpened():
        _ok, _live_frame = _capture.read()
        _capture.release()
        _source = _live_frame if _ok else None
    else:
        _capture.release()
        _source = None
    if _source is None:
        print('{"event": "error", "code": "CAMERA_UNAVAILABLE", "message": "live cell needs a camera"}')
    else:
        _model, _report = load_pose_model()
        print(_report)
        _persons = infer_frame(_model, _source)
        print(f"persons: {len(_persons)}")
        for _person in _persons:
            print(_person["xyn"].shape, _person["visibility"].mean())